# Hands On: how does a fake RAG work

In this Hands-on we will see what is a RAG and how does it work. We can imagine this RAG as small version of
a LLM. We are using a **fake** model to quickly have a look to the model 

**What IS meaningful with `fake`:**
- the full pipeline mechanics: LCEL chain, Chroma ingest/persist, retriever, citations;
- `n_chunks` at ingest (it responds to your `chunk_size` / `chunk_overlap` choices);
- the **guardrail** (the fake LLM is rule-based: out-of-corpus question → honest refusal);
- **groundedness** and **latency** of the canned answers.

**What is NOT meaningful with `fake`:** the deterministic fake embeddings carry
  **no semantics** (similar texts are not necessarily close), so these metrics stay ~0
  and do not respond to `k` or to question rephrasing. It matches the question against a fixed vocabulary instead.


In [45]:
from pathlib import Path
from pydantic_settings import BaseSettings, SettingsConfigDict


# The notebook lives in the project root: make the location explicit.
PROJECT_ROOT = Path.cwd()
print("PROJECT_ROOT:", PROJECT_ROOT)

# (Alternative: point manually to the root if you run the notebook from another folder)
# PROJECT_ROOT = Path("/path/to/project")

# This class builds the default for the parameters.


class Settings(BaseSettings):
    model_config = SettingsConfigDict(
        env_file=".env",
        env_prefix="COPILOT_",
        extra="ignore",
    )


    # --- Paths ---
    # PDF EDITION: the documents to index live in data/pdf/ (no .md)
    pdf_dir: Path = PROJECT_ROOT / "data" / "pdf"
    chroma_dir: Path = PROJECT_ROOT / "data" / "chroma"
    collection_name: str = "maintenance_manuals"

    # --- RAG parameters ---
    chunk_size: int = 800          # characters per chunk
    chunk_overlap: int = 120       # overlap between consecutive chunks
    retrieval_k: int = 4           # how many chunks to retrieve per question


# Single shared instance imported across the app.
settings = Settings()

PROJECT_ROOT: /home/mpezzella/Desktop/hw.pezzella.2026


A single variable controls everything: **run this cell before doing anything else** (and re-run the subsequent cells if you change it after starting the project).

- `"fake"` — no installations required: the embedding is deterministic (without semantic understanding), and the LLM is a small mock model that **reads the question** and responds based on the corpus (a question about the PDF → an answer based on LEVEL facts, with a citation; an unrelated question → *"I don't know"*). Useful for testing **the entire pipeline** (LCEL, Chroma, retriever, tests) without downloading anything, and for running the notebook in CI.

In [46]:
import os

BACKEND = os.environ.get("COPILOT_BACKEND", "fake")


| Parameter | Default | What it controls |
|---|---|---|
| `chunk_size` | 800 | characters per chunk: small = fine index, large = whole context |
| `chunk_overlap` | 120 | characters shared between consecutive chunks (protect facts that straddle the cut) |
| `retrieval_k` | 4 | how many chunks to retrieve per question |

**AFTER each change**: re-run this cell, then INGEST (§10) and EVALUATION (§11),
and record the results in a table inside the notebook.


In [47]:
STUDENT = "Guy Smith"

settings.chunk_size    =  800        
settings.chunk_overlap =  120        
settings.retrieval_k   =    4        

print("Student:", STUDENT)
print("chunk_size =", settings.chunk_size, "| chunk_overlap =", settings.chunk_overlap,
      "| retrieval_k =", settings.retrieval_k)

# Invalidate the caches: the retriever and the chain must be rebuilt with the NEW
# parameters (same mechanism used by the backend-switch cells).
for _name in ("get_vectorstore", "get_retriever", "get_rag_chain"):
    _f = globals().get(_name)
    if hasattr(_f, "cache_clear"):
        _f.cache_clear()

# If you changed chunk_size/overlap: the on-disk index still holds the OLD chunks.
# Remember to re-run the INGEST cell of section 10 (ingest(reset=True)).


Student: Guy Smith
chunk_size = 800 | chunk_overlap = 120 | retrieval_k = 4


Run **after choosing the backend** (the dependencies to check depend on `BACKEND`):
it imports nothing heavy, checks with `importlib.util.find_spec` whether each package
is already present in the environment and otherwise installs it via pip. With the
`fake` backend a clean instance downloads only what the pipeline actually uses.


In [48]:
import importlib.util
import subprocess
import sys

# module_name(import) -> package_name(pip). The check uses find_spec, which does NOT
# execute the module (also avoiding the heavy import warnings of langchain-community).
PIP_URL = "https://pypi.org/project/{pkg}/"
DEPENDENCIES = {
    "fake": {   # common base: full pipeline without models
        "langchain_core": "langchain-core",
        "langchain_text_splitters": "langchain-text-splitters",
        "langchain_community": "langchain-community",
        "langchain_chroma": "langchain-chroma",
        "chromadb": "chromadb",
        "pydantic_settings": "pydantic-settings",
        "pypdf": "pypdf",
        "sklearn": "scikit-learn",
        "numpy": "numpy",
    }
}

required = dict(DEPENDENCIES.get("fake", {}))   # base: always
required.update(DEPENDENCIES.get(BACKEND, {}))  # plus the chosen backend's extras

missing = [pkg for mod, pkg in required.items() if importlib.util.find_spec(mod) is None]

if not missing:
    print(f"All {len(required)} required dependencies are already installed ✓")
else:
    print("Missing packages:", ", ".join(missing))
    print(" --> details:", [PIP_URL.format(pkg=p) for p in missing])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    print("Installation complete ✓")

print("Selected backend:", BACKEND, "| dependencies checked:", len(required))


All 9 required dependencies are already installed ✓
Selected backend: fake | dependencies checked: 9


In [49]:
if BACKEND == "fake":
    # ------------------------------------------------------------------
    # 3.1 BACKEND "fake" — no installs, no downloads.
    # ------------------------------------------------------------------
    from langchain_core.embeddings import DeterministicFakeEmbedding
    from langchain_core.language_models.chat_models import BaseChatModel
    from langchain_core.messages import AIMessage
    from langchain_core.outputs import ChatGeneration, ChatResult

    EMBEDDING_DIM = 384
    print("Loading the fake backend...")

    class FakeRagChatModel(BaseChatModel):
        """Mini rule-based LLM: it reads the QUESTION (not the retrieved context) and
        *deliberately* decides whether to answer from the corpus or admit it doesn't
        know. The verdict is based on the same mechanism as the real guardrail: if the
        question's terms don't belong to the document's domain (HERE: the LEVEL PDF),
        the honest answer is 'I don't know'. So the final in/out-of-corpus test stays
        meaningful even without any real model."""
        corpus_answer: str
        out_of_corpus_answer: str
        corpus_words: tuple = (
            "schrodinger", "level", "rmin", "rmax", "mesh", "eigenvalue",
            "convergence", "integration", "input",
            # NB: words like 'eps'/'rh' are too short/generic and are excluded:
            # 'eps' would match as a substring of other English terms.
        )

        @property
        def _llm_type(self) -> str:
            return "fake-rag-chat"

        def _generate(self, messages, stop=None, run_manager=None, **kwargs):
            text = str(messages[-1].content).lower()
            # isolate the QUESTION: the guardrail evaluates the user's question, NOT the
            # retrieved context (which always comes from the document, so it cannot
            # decide whether the question is in-corpus).
            question = text.rsplit("question:", 1)[-1]
            out_of_corpus = not any(p in question for p in self.corpus_words)
            content = self.out_of_corpus_answer if out_of_corpus else self.corpus_answer
            return ChatResult(generations=[ChatGeneration(message=AIMessage(content=content))])

    fake_llm = FakeRagChatModel(
        # canned answers aligned with the REAL facts of the sample PDF (appendix B:
        # definitions of RH = mesh size and EPS = eigenvalue convergence parameter)
        corpus_answer=(
            "RH is the mesh size used to build the numerical integration mesh over the "
            "[RMIN, RMAX] interval, while EPS is the eigenvalue convergence parameter "
            "used by SCHRQ (in cm-1). (source: level16documentation.pdf)"
        ),
        out_of_corpus_answer=(
            "I don't know: the retrieved PDF context does not contain this information, "
            "so I cannot answer without inventing."
        ),
    )

    # Fake BUT deterministic embedding: same text -> same vector.
    # WARNING: there is no real semantics -- similar sentences are NOT necessarily
    # close, so retrieval is not meaningful (it's a mechanics test).
    fake_embeddings = DeterministicFakeEmbedding(size=EMBEDDING_DIM)

    def get_llm():
        return fake_llm

    def get_embeddings():
        return fake_embeddings

    print("FAKE backend ready: embedding", EMBEDDING_DIM,
          "dim, canned answers for in/out-of-corpus questions.")

    # The on-disk index contains vectors generated by the previous model: pointing to
    # a different directory avoids corrupting it (and it is rebuilt with the new backend).
    settings.chroma_dir = PROJECT_ROOT / "data" / f"chroma-{BACKEND}"
    print("chroma_dir ->", settings.chroma_dir)

    # Invalidate components already built with another backend (if cached).
    for _name in ("get_vectorstore", "get_retriever", "get_rag_chain", "get_llm", "get_embeddings"):
        _f = globals().get(_name)
        if hasattr(_f, "cache_clear"):
            _f.cache_clear()


Loading the fake backend...
FAKE backend ready: embedding 384 dim, canned answers for in/out-of-corpus questions.
chroma_dir -> /home/mpezzella/Desktop/hw.pezzella.2026/data/chroma-fake


Chroma is a *vector database*: it stores the embedding vectors of our document chunks
(together with their original text and metadata) and, given a query vector, returns the
 most semantically similar chunks. This is the "R" (retrieval) in RAG.

 Both sides of the app go through this one function:
   - ingestion writes chunks into the store;
   - the retriever reads the most relevant chunks back out at query time.

 Keeping it in one place guarantees they use the SAME collection, the SAME on-disk
 location, and the SAME embedding function — the three things that must match for
 retrieval to work at all.


In [50]:
from functools import lru_cache

from langchain_chroma import Chroma


@lru_cache
def get_vectorstore() -> Chroma:
    """Return the shared Chroma vector store (built once, then reused).

    The store is persisted to ``settings.chroma_dir`` on disk, so the index survives
    restarts: we ingest the documents once and can query them from any later run.
    """
    # Chroma's persistent client creates this directory on first use, but we make sure
    # it exists so the very first run never fails on a missing path.
    settings.chroma_dir.mkdir(parents=True, exist_ok=True)

    return Chroma(
        collection_name=settings.collection_name,
        # The embedding function is used to vectorize incoming queries. It MUST be the
        # same model used at ingestion time, otherwise the vectors are not comparable.
        embedding_function=get_embeddings(),
        persist_directory=str(settings.chroma_dir),
    )



Turning raw PDFs into something we can retrieve happens in three steps:

  1. **load** — read every PDF under `data/pdf/` into LangChain `Document` objects with
     `PyPDFLoader`: **1 page = 1 Document** (a PDF is paginated by construction);
  2. **split** — cut each document into overlapping chunks small enough to embed and to
     fit into a prompt;
  3. **index** — store the chunks in Chroma, which embeds each one and saves the vector.


In [51]:
from pathlib import Path

from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter


def load_documents() -> list[Document]:
    """Load every PDF under settings.pdf_dir, ONE Document per page.

    PyPDFLoader parses with pypdf and returns one Document per page, already carrying
    metadata["source"] (full path) and metadata["page"] (0-based). For citations we
    keep only the FILE NAME, and the page becomes 1-based.
    """
    pdfs = sorted(settings.pdf_dir.glob("**/*.pdf"))
    if not pdfs:
        raise FileNotFoundError(
            f"No PDF in {settings.pdf_dir} — put the document to index there "
            "(section 10 prepares the sample one automatically).")

    documents: list[Document] = []
    for pdf in pdfs:
        documents.extend(PyPDFLoader(str(pdf)).load())

    for doc in documents:
        doc.metadata["source"] = Path(doc.metadata["source"]).name
        doc.metadata["page"] = int(doc.metadata.get("page", 0)) + 1

    return documents


def split_documents(documents: list[Document]) -> list[Document]:
    """Split documents into overlapping chunks using sizes from config.

    ``RecursiveCharacterTextSplitter`` tries a list of separators in order (paragraphs,
    then lines, then sentences, then words), so it keeps semantic units together as much
    as the chunk size allows. The overlap carries a bit of context across the cut, so a
    fact that sits on a chunk boundary is not lost.
    """
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=settings.chunk_size,
        chunk_overlap=settings.chunk_overlap,
        add_start_index=True,  # record each chunk's character offset in the source doc
    )
    return splitter.split_documents(documents)


def ingest(reset: bool = True) -> int:
    """Load, split, and index the PDFs in Chroma. Return the number of chunks stored.

    When ``reset`` is True the collection is emptied first, so re-running ingestion does
    not create duplicate chunks — the operation is idempotent.
    """
    documents = load_documents()
    chunks = split_documents(documents)

    vectorstore = get_vectorstore()
    if reset:
        vectorstore.reset_collection()

    # add_documents embeds each chunk (via the store's embedding function) and persists it.
    vectorstore.add_documents(chunks)

    return len(chunks)


 A *retriever* is LangChain's standard interface for "given a question, return the most relevant documents". We build ours from the Chroma vector store.

 Why wrap the store in a retriever instead of calling ``similarity_search`` directly?
   - Composability: a retriever is a LangChain ``Runnable``, so it plugs straight into
     chains (and later into LangGraph nodes) without manual glue.
   - Abstraction: it hides *how* documents are fetched. Today that is cosine similarity
     in Chroma; swapping in re-ranking, hybrid search, or MMR later would not change any
     code downstream — exactly the reasoning behind ``get_llm()`` and ``get_embeddings()``.


In [52]:
from functools import lru_cache

from langchain_core.vectorstores import VectorStoreRetriever


@lru_cache
def get_retriever() -> VectorStoreRetriever:
    """Return the shared retriever (built once, then reused).

    ``k`` (how many chunks to fetch per question) comes from config and must be passed
    inside ``search_kwargs`` — a common gotcha with ``as_retriever``.
    """
    return get_vectorstore().as_retriever(
        search_kwargs={"k": settings.retrieval_k},
    )


The input (the question string) feeds a dict that builds the two template variables in parallel — ``context`` by running the retriever and formatting its chunks, and ``question`` by passing the input through unchanged. That dict fills the PROMPT, the PROMPT goes to the LLM, and StrOutputParser extracts the plain text of the reply.

The prompt contains guardrails that the llm + RAG model will apply, it guarantees the groundedness 
and avodi allucinations. 

  - Write a *healty* prompt, that answers only from the context, it is honest (answers "I don't know" if answer is unknown), it concise and cites the sources.
  - Write an *unhealty* source (do your worst), and try to see how does it work in references to the points above (not with fake).

In [53]:
from functools import lru_cache

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import Runnable, RunnablePassthrough


# TODO write me 
SYSTEM_PROMPT = (    "You are a computational chemistry student. Answer the user's question using only "
    "the context below, which is the documentation of the LEVEL program (bound and "
    "quasibound levels).\n"
    "- If the context does not contain the answer, say that you don't know: never invent.\n"
    "- Be concise and practical.\n"
    "- Cite the source document(s) you used, e.g. (source: document.pdf)."
)

PROMPT = ChatPromptTemplate.from_messages(
    [
        ("system", SYSTEM_PROMPT),
        ("human", "Context:\n{context}\n\nQuestion: {question}"),
    ]
)


def format_docs(docs) -> str:
    """Turn retrieved Document objects into a single context string for the prompt.

    Each chunk is prefixed with its source file so the model can cite it, and chunks are
    separated by blank lines so they read as distinct passages.
    """
    return "\n\n".join(
        f"[Source: {doc.metadata['source']}]\n{doc.page_content}"
        for doc in docs
    )


@lru_cache
def get_rag_chain() -> Runnable:
    """Build the linear RAG chain (built once, then reused)."""
    return (
        {
            "context": get_retriever() | format_docs,
            "question": RunnablePassthrough(),
        }
        | PROMPT
        | get_llm()
        | StrOutputParser()
    )


def answer(question: str) -> str:
    """Convenience wrapper: run the RAG chain on a question and return the answer text."""
    return get_rag_chain().invoke(question)


# Invalidate the cache: the next get_rag_chain() builds with the NEW prompt.
# (On the first pass get_rag_chain does not exist yet: the guard avoids the error.)
_f = globals().get("get_rag_chain")
if hasattr(_f, "cache_clear"):
    _f.cache_clear()


First the `data/pdf/` folder is prepared: if it is empty, the notebook copies the sample
PDF there from a custom position. Then the index is built and a real question is asked.

- With `BACKEND="fake"`: the answer is *canned* but coherent (citation and guardrail work
  by construction) — the value is in verifying the **mechanics**.



We are reading the manual  of the code Level from Bob Leroy, used for generating molecular spectra of diatomics. 


In [54]:
import shutil

settings.pdf_dir.mkdir(parents=True, exist_ok=True)

!wget https://uwaterloo.ca/atmospheric-chemistry-experiment/sites/default/files/uploads/documents/level16documentation.pdf

PDF_EXAMPLE = Path("level16documentation.pdf") 

if not list(settings.pdf_dir.glob("*.pdf")) and PDF_EXAMPLE.exists():
    shutil.copy(PDF_EXAMPLE, settings.pdf_dir / PDF_EXAMPLE.name)
    print("copied:", PDF_EXAMPLE, "->", settings.pdf_dir)

pdf_files = sorted(settings.pdf_dir.glob("**/*.pdf"))
print("PDFs in", settings.pdf_dir, ":", [p.name for p in pdf_files])
assert pdf_files, "data/pdf/ is empty and the sample PDF is not found: add one."


--2026-11-24 14:22:22--  https://uwaterloo.ca/atmospheric-chemistry-experiment/sites/default/files/uploads/documents/level16documentation.pdf
Resolving uwaterloo.ca (uwaterloo.ca)... 151.101.194.132, 151.101.2.132, 151.101.66.132, ...
Connecting to uwaterloo.ca (uwaterloo.ca)|151.101.194.132|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 310984 (304K) [application/pdf]
Saving to: ‘level16documentation.pdf.2’

level16documentatio 100%[===================>] 303.70K   987KB/s    in 0.3s    

2026-11-24 14:22:22 (987 KB/s) - ‘level16documentation.pdf.2’ saved [310984/310984]

copied: level16documentation.pdf -> /home/mpezzella/Desktop/hw.pezzella.2026/data/pdf
PDFs in /home/mpezzella/Desktop/hw.pezzella.2026/data/pdf : ['level16documentation.pdf']


In [55]:
# TODO re-run this cell every time you change somehting in excercise A
n_chunks = ingest(reset=True)
print("Indexed chunks:", n_chunks)
print("(current parameters: chunk_size =", settings.chunk_size,
      "| chunk_overlap =", settings.chunk_overlap, "| k =", settings.retrieval_k, ")")


Indexed chunks: 225
(current parameters: chunk_size = 800 | chunk_overlap = 120 | k = 4 )



Write down a question for the RAG that is meaningful for the article, and, down below define the *gold* answer (what is expected) and the *expected source*.

Ask the same question in "sparse" form (failure of the linear RAG). A question not aligned with the document's vocabulary does not retrieve the gold chunk.


In [56]:
question = ("What is EPS, the eigenvalue convergence parameter, and what is "
            "the RH mesh size used for in LEVEL?")
answer_text = answer(question)
print(answer_text)


RH is the mesh size used to build the numerical integration mesh over the [RMIN, RMAX] interval, while EPS is the eigenvalue convergence parameter used by SCHRQ (in cm-1). (source: level16documentation.pdf)


In [57]:
# =====================================================================
# EXPERIMENT REGISTRY — your lab notebook
# =====================================================================
# After EVERY evaluation record the run: log_experiment("<exercise id>", m, note="...")
# `m` is the dict computed by evaluate_rag(...) and printed by _print_metrics(...).
# At the end of the lab: PRINT_REGISTRY() for the comparison table of all runs.

REGISTRY = []


def log_experiment(name: str, m: dict, note: str = "") -> dict:
    """Append one row to the REGISTRY with the parameters and metrics of a run."""
    row = {
        "exercise": name,
        "student": globals().get("STUDENT", "?"),
        "backend": BACKEND,
        "chunk": f"{settings.chunk_size}/{settings.chunk_overlap}",
        "k": m.get("k"),
        "recall@k": m.get("recall@k"),
        "precision@k": m.get("precision@k"),
        "mrr": m.get("mrr"),
        "relevance": None if BACKEND == "fake" else m.get("relevance"),
        "groundedness": m.get("groundedness"),
        "guardrail": m.get("guardrail_ok"),
        "t_retrieve": round(m.get("t_retrieve", float("nan")), 3),
        "t_generate": round(m.get("t_generate", float("nan")), 3),
        "note": note,
    }
    REGISTRY.append(row)
    print(f"[registry] run recorded: {name!r} (total: {len(REGISTRY)})")
    return row


def _fmt(v, pct=True):
    return "–" if v is None else (f"{v:.2f}" if pct else str(v))


def PRINT_REGISTRY():
    """Comparison table of all runs recorded so far."""
    if not REGISTRY:
        print("REGISTRY empty: call log_experiment(...) after every evaluation.")
        return
    header = (f"{'#':<3}{'exercise':<20}{'student':<14}{'backend':<8}{'chunk':<10}"
              f"{'k':<3}{'recall@k':<9}{'prec@k':<8}{'mrr':<6}{'relv':<6}"
              f"{'ground':<8}{'guard':<7}{'note'}")
    print(header)
    print("-" * len(header))
    for i, r in enumerate(REGISTRY, 1):
        print(f"{i:<3}{r['exercise'][:19]:<20}{r['student'][:13]:<14}{r['backend']:<8}"
              f"{r['chunk']:<10}{r['k']:<3}{_fmt(r['recall@k']):<9}{_fmt(r['precision@k']):<8}"
              f"{_fmt(r['mrr']):<6}{_fmt(r['relevance']):<6}{_fmt(r['groundedness']):<8}"
              f"{str(r['guardrail']):<7}{r['note'][:34]}")


print("Registry ready: record every exercise with log_experiment(...).")


Registry ready: record every exercise with log_experiment(...).


### Evaluation — quantitative metrics on a real question

Every time you ask a question about a document, `evaluate_rag()` runs the complete
**retrieve → answer** cycle and measures quality with standard industry metrics,
computed autonomously (no external judges, no judge APIs):

**Retrieval** (on the chunks actually returned, against the corpus's "gold" content):
| Metric | Question it answers |
|---|---|
| `Recall@k` | of all the gold chunks that exist in the corpus, how many appear in the first k retrieved? |
| `Precision@k` | of the k retrieved chunks, how many are really gold? |
| `MRR` (Mean Reciprocal Rank) | how high does the first gold chunk rank in the retrieval list? (1.0 = first result) |
| `fact retrieved` | does the *expected fact* (e.g. "mesh size") appear in one of the retrieved chunks? |

**Answer** (on the generated answer):
| Metric | Question it answers |
|---|---|
| `relevance` (question↔answer cosine) | does the answer really talk about what I asked? — *requires real semantic embedding (local/ollama); with `fake` it is not meaningful* |
| `groundedness` (token overlap) | how much of the answer's text is literally in the retrieved context? (LLM-free proxy of RAGAS's *faithfulness*) |
| `guardrail` | does the answer admit it doesn't know when the corpus doesn't cover the question? |
| `cites expected source` | does the answer cite the expected document in the `(source: ...)` tags? |

**Efficiency**: `t_retrieve` and `t_generate` in seconds (the latter includes the
retrieval re-run internally by the LCEL chain).

Here the gold facts really exist in the corpus (appendix B of the PDF defines
`RH = mesh size` and `EPS = eigenvalue convergence parameter`): the retrieval metrics
are therefore **meaningful** with real backends.


In [58]:
import re
import time
import numpy as np


def _clean_tokens(s: str) -> set:
    """Lowercase tokens, without stopwords or one-letter words."""
    stop = {"the", "a", "an", "to", "of", "and", "or", "is", "are", "in", "for",
            "your", "you", "it", "this", "that", "i", "be", "on", "at", "as", "use"}
    return {t for t in re.findall(r"[a-z0-9]+", s.lower())
            if t not in stop and len(t) > 1}


def _cosine(a, b) -> float:
    a, b = np.asarray(a, float), np.asarray(b, float)
    na, nb = np.linalg.norm(a), np.linalg.norm(b)
    return float(a @ b / (na * nb)) if na and nb else 0.0


DONT_KNOW_SIGNALS = ["don't know", "do not know", "not contain", "no information",
                     "cannot", "can't", "not available", "not provide", "not in the",
                     "non lo so"]


def evaluate_rag(query: str, expected_facts: list = None, expected_source: str = None,
                 k: int = None) -> dict:
    """Run the query on the full pipeline (retrieve + answer) and compute the
    evaluation metrics described above.

    expected_facts : substrings identifying the "gold" content
                     (e.g. ["mesh size", "eigenvalue convergence parameter"]); a chunk
                     is gold if it contains at least one. Optional (out-of-corpus
                     questions).
    expected_source : the .pdf file name the answer should cite.
    """
    expected_facts = expected_facts or []
    k = k or settings.retrieval_k
    m = {"query": query, "k": k}

    # ---------- RETRIEVAL ----------
    t0 = time.perf_counter()
    docs = get_retriever().invoke(query)
    m["t_retrieve"] = time.perf_counter() - t0

    if expected_facts:
        total_gold = 0
        for chunk in get_vectorstore()._collection.get()["documents"]:
            if any(f.lower() in chunk.lower() for f in expected_facts):
                total_gold += 1
        m["gold_chunks_in_corpus"] = total_gold

        ranks = [i + 1 for i, d in enumerate(docs)
                 if any(f.lower() in d.page_content.lower() for f in expected_facts)]
        n_retrieved_gold = len(ranks)
        m["recall@k"]    = n_retrieved_gold / total_gold if total_gold else None
        m["precision@k"] = n_retrieved_gold / k
        m["mrr"]         = 1 / ranks[0] if ranks else 0.0
        m["fact_retrieved"] = any(f.lower() in d.page_content.lower()
                                  for d in docs for f in expected_facts)
    else:
        m["recall@k"] = m["precision@k"] = m["mrr"] = None   # no gold defined
        m["fact_retrieved"] = None

    # ---------- GENERATION ----------
    t0 = time.perf_counter()
    answer_text = answer(query)
    m["t_generate"] = time.perf_counter() - t0

    context = format_docs(docs)

    # relevance: requires embeddings with real semantics (local/ollama)
    v_q = get_embeddings().embed_query(query)
    v_a = get_embeddings().embed_query(answer_text)
    m["relevance"] = _cosine(v_q, v_a)

    # groundedness: token-overlap answer vs context (LLM-free faithfulness)
    answer_toks = _clean_tokens(answer_text)
    m["groundedness"] = (len(answer_toks & _clean_tokens(context)) / len(answer_toks)
                         if answer_toks else 0.0)

    # guardrail: does the answer admit it doesn't know?
    low = answer_text.lower()
    m["guardrail_ok"] = any(s in low for s in DONT_KNOW_SIGNALS)

    # citation of the expected source (the ingestion .pdf files)
    cited_sources = re.findall(r"source\s*:\s*([\w .\-]+?\.pdf)", low, re.I)
    m["cited_sources"] = cited_sources
    m["cites_expected_source"] = bool(expected_source) and expected_source in cited_sources

    m["answer"] = answer_text
    return m


def _print_metrics(m: dict):
    def fmt(v, pct=False):
        if v is None:
            return "–  (not computable)"
        return f"{v:.1%}" if pct else f"{v:g}"
    print(f"QUERY : {m['query']!r}\n      k = {m['k']}")
    print(f"{'metric':<24}{'value':<28}{'interpretation'}")
    print("-" * 88)
    if "gold_chunks_in_corpus" in m:
        print(f"{'gold chunks in corpus':<24}{m['gold_chunks_in_corpus']:<28}content retrieval should aim at")
    print(f"{'recall@k':<24}{fmt(m['recall@k'], pct=m['recall@k'] is not None):<28}gold chunks in the first {m['k']}")
    print(f"{'precision@k':<24}{fmt(m['precision@k'], pct=m['precision@k'] is not None):<28}how much of the retrieval is gold")
    print(f"{'mrr':<24}{fmt(m['mrr']):<28}position of the first gold chunk (1.0 = ideal)")
    print(f"{'fact retrieved':<24}{str(m['fact_retrieved']):<28}does the expected fact appear in a chunk?")
    if BACKEND == "fake":
        print(f"{'relevance':<24}{m['relevance']:<28.1%} fake embedding: NOT meaningful")
    else:
        print(f"{'relevance':<24}{m['relevance']:<28.1%}question/answer semantics, higher = better")
    print(f"{'groundedness':<24}{m['groundedness']:<28.1%}answer anchored to the context? (1.0 = ideal)")
    print(f"{'guardrail':<24}{str(m['guardrail_ok']):<28}can say 'I don't know' when needed")
    print(f"{'cites expected source':<24}{str(m['cites_expected_source']):<28}sources found in the answer: {m['cited_sources']}")
    print(f"{'t_retrieve':<24}{m['t_retrieve']:.2f}s{'':<12}retrieval from the vector store")
    print(f"{'t_generate':<24}{m['t_generate']:.2f}s{'':<12}LCEL chain end-to-end (incl. 2nd retrieval)")
    print("ANSWER:", m["answer"][:160].replace("\n", " "), "...")


In [ ]:
GOLD_FACTS = ["mesh size", "eigenvalue convergence parameter"]
EXPECTED_SOURCE = "level16documentation.pdf"

m_in_corpus = evaluate_rag(question, expected_facts=GOLD_FACTS, expected_source=EXPECTED_SOURCE)
_print_metrics(m_in_corpus)

# quantitative checks: every metric is in [0, 1] (or None, if gold is not defined);
# the only exception is 'relevance', which is a cosine and lies in [-1, 1]
for name in ["recall@k", "precision@k", "mrr", "groundedness"]:
    v = m_in_corpus[name]
    if v is not None:
        assert 0.0 <= v <= 1.0, f"{name} out of range: {v}"
assert -1.0 <= m_in_corpus["relevance"] <= 1.0
assert m_in_corpus["t_retrieve"] >= 0 and m_in_corpus["t_generate"] >= 0
print("\n[assert] metrics valid ✓")


QUERY : 'What is EPS, the eigenvalue convergence parameter, and what is the RH mesh size used for in LEVEL?'
      k = 4
metric                  value                       interpretation
----------------------------------------------------------------------------------------
gold chunks in corpus   1                           content retrieval should aim at
recall@k                0.0%                        gold chunks in the first 4
precision@k             0.0%                        how much of the retrieval is gold
mrr                     0                           position of the first gold chunk (1.0 = ideal)
fact retrieved          False                       does the expected fact appear in a chunk?
relevance               5.7%                         fake embedding: NOT meaningful
groundedness            40.9%                       answer anchored to the context? (1.0 = ideal)
guardrail               False                       can say 'I don't know' when needed
cites expect

In [65]:
sparse_question = "What is RH and what is EPS used for?"
m_sparse = evaluate_rag(sparse_question, expected_facts=GOLD_FACTS, expected_source=EXPECTED_SOURCE)
print(">>> metrics with the 'sparse' question (the push for the agentic phase):")
_print_metrics(m_sparse)

if BACKEND != "fake":
    print("\n[comparison] aligned:", m_in_corpus["recall@k"],
          "| sparse:", m_sparse["recall@k"],
          "-> if the sparse one is lower, here is the 'inconvenience' to fix in phase 2.")


>>> metrics with the 'sparse' question (the push for the agentic phase):
QUERY : 'What is RH and what is EPS used for?'
      k = 4
metric                  value                       interpretation
----------------------------------------------------------------------------------------
gold chunks in corpus   1                           content retrieval should aim at
recall@k                0.0%                        gold chunks in the first 4
precision@k             0.0%                        how much of the retrieval is gold
mrr                     0                           position of the first gold chunk (1.0 = ideal)
fact retrieved          False                       does the expected fact appear in a chunk?
relevance               4.5%                         fake embedding: NOT meaningful
groundedness            7.1%                        answer anchored to the context? (1.0 = ideal)
guardrail               True                        can say 'I don't know' when needed
c

What happens when we ask a new questions OUTSIDE the corpus. Does the guardrail hold for ANY question, or does it depend on how it is written?


In [66]:
# --- EVALUATION: question OUTSIDE the corpus (no gold) ---
# here the value lies in the 'guardrail' and latency — recall/precision make no sense.
m_out_of_corpus = evaluate_rag("What is the correct tire pressure for a road bicycle?")
_print_metrics(m_out_of_corpus)

# the only right behavior: admit you don't know
if m_out_of_corpus["guardrail_ok"]:
    print("\n[assert] anti-hallucination guardrail confirmed by the metric ✓")
elif BACKEND == "fake":
    raise AssertionError("GUARDRAIL FAILED with the fake backend!")
else:
    print("\n guardrail NOT respected by the real LLM (answer not grounded in the "
          "context): behavior to measure and strengthen, exactly what the "
          "agentic phase (check/regenerate) must correct.")



QUERY : 'What is the correct tire pressure for a road bicycle?'
      k = 4
metric                  value                       interpretation
----------------------------------------------------------------------------------------
recall@k                –  (not computable)         gold chunks in the first 4
precision@k             –  (not computable)         how much of the retrieval is gold
mrr                     –  (not computable)         position of the first gold chunk (1.0 = ideal)
fact retrieved          None                        does the expected fact appear in a chunk?
relevance               11.4%                        fake embedding: NOT meaningful
groundedness            7.1%                        answer anchored to the context? (1.0 = ideal)
guardrail               True                        can say 'I don't know' when needed
cites expected source   False                       sources found in the answer: []
t_retrieve              0.02s            retrieval from t